# NF-SingleEvent Prior Test

This new paper outlines how to re-calculate likelihoods and priors, including code: https://zenodo.org/records/18337354

The Phenom analysis pipeline includes both log_likelihoods **and** priors, but the Mixed and SEOBNR pipeline only contains the log_likelihoods. However, the analytical priors should be the same:
1. Test whether the Phenom posterior samples (i.e. the parameters) include exactly the SEOBNR parameters -> could then just use those priors (that does not make any sense, forget it)
2. Confirm whether my own prior model from a O3 Emulator training / detection efficiency calculation spits out the same numbers (relationships)

### Imports

In [31]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy.special import logit, expit

import matplotlib.pyplot as plt
import corner

import jax
import jax.numpy as jnp
import optax
import equinox as eqx

# FlowJAX (new API)
from flowjax.flows import masked_autoregressive_flow
from flowjax.distributions import Normal

from pathlib import Path

import seaborn as sns
from tqdm import tqdm, trange
import utils as ut

import pesummary as pes
from pesummary.io import read
import bilby

from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
import astropy.constants as constants

import matplotlib as mpl
mpl.rcParams["text.usetex"] = False

# silence unnecessary warnings about some specific model not being available (shouldn't hurt performance according to ChatGPT)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import os
import socket
print(f"Hostname: {socket.gethostname()}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
!nvidia-smi

Hostname: th-cl-edger731.hpc.physik.uni-muenchen.de
CUDA_VISIBLE_DEVICES: -1
/bin/bash: line 1: nvidia-smi: command not found


In [3]:
jax.devices()

ERROR:2026-02-02 15:54:09,416:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/j/Julius.Gassert/.conda/envs/jax/lib/python3.14/site-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/j/Julius.Gassert/.conda/envs/jax/lib/python3.14/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/j/Julius.Gassert/.conda/envs/jax/lib/python3.14/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load 

[CpuDevice(id=0)]

In [32]:
# Assumed constants
H0cosmo = 67.9 # constants from https://dcc.ligo.org/public/0170/P2000318/011/o3b_catalog.pdf
Om0cosmo = 0.3065
cosmo = FlatLambdaCDM(H0=H0cosmo * u.km / u.s / u.Mpc, Om0=Om0cosmo)
speed_of_light = constants.c.to(u.km / u.s).value

def ddL_of_z(z, dL, H0):
    return dL / (1 + z) + speed_of_light * (1 + z) / (H0 * cosmo.efunc(z))

In [33]:
mass_grid = np.linspace(1,1000,10000)

def conditional_powerlaw_pdf(m2, m1, min_m2, alpha, conditional):
    p = np.zeros_like(m2)

    if conditional:
        valid = (m2 >= min_m2) & (m2 <= m1)
    else:
        raise NotImplementedError

    if alpha == -1:
        norm = np.log(m1 / min_m2)
        p[valid] = 1.0 / (m2[valid] * norm[valid])
    else:
        norm = (m1**(alpha + 1) - min_m2**(alpha + 1)) / (alpha + 1)
        p[valid] = m2[valid]**alpha / norm[valid]

    return p

def bounded_powerlaw_pdf(m1,min_m1, max_m1, alpha_m1, ref_m1):
    pm1 = (mass_grid - ref_m1)**alpha_m1
    sel_above = mass_grid > max_m1
    sel_below = mass_grid < min_m1
    sel = (sel_above | sel_below)
    pm1[sel] = 0
    pm1 /= np.trapezoid(pm1, mass_grid)
    return interp1d(mass_grid, pm1, fill_value = 0, bounds_error = False)(m1)#(m1-ref_m1)

def redshift_pdf(z, z_grid, dVdz_grid, kappa):
    num_grid = dVdz_grid * (1 + z_grid)**(kappa-1)
    den = np.trapezoid(num_grid, z_grid)
    num_grid /= den
    
    return interp1d(z_grid, num_grid, fill_value = 0, bounds_error = False)(z)

def model_pdf_gwtc3_arr(data, population_params):
    # model here: https://zenodo.org/records/5636816
    min_m1 = population_params["min_m1"]
    max_m1 = population_params["max_m1"]
    alpha_m1 = population_params["alpha_m1"]
    
    ref_m1 = population_params["ref_m1"]
    
    min_m2 = population_params["min_m2"]
    max_m2 = population_params["max_m2"]
    alpha_m2 = population_params["alpha_m2"]
    
    max_a1 = population_params["max_a1"]
    max_a2 = population_params["max_a2"]

    kappa = population_params["kappa"]
    conditional_mass = population_params["conditional_mass"]

    m1 = data["m1_source"]
    m2 = data["m2_source"]
    a1 = data["a1"]
    a2 = data["a2"]
    z = data["redshift"]

    z_grid = np.linspace(0, population_params["zMax"], 10000)
    dVdz_grid = cosmo.differential_comoving_volume(z_grid).to(u.Gpc**3 / u.sr).value * 4 * np.pi  # full sky

    pm1 = bounded_powerlaw_pdf(m1, min_m1, max_m1, alpha_m1, ref_m1)

    # calculate for m2
    if conditional_mass:
        pm2 = conditional_powerlaw_pdf(m2, m1, min_m2, alpha_m2, conditional_mass)
    else:
        pm2 = bounded_powerlaw_pdf(m2, min_m2, max_m2, alpha_m2, 0)

    # spin magnitudes
    a1_sel, a2_sel = a1 > max_a1, a2 > max_a2
    
    pa1 = 1/(4*np.pi*(a1**2)*max_a1)
    pa2 = 1/(4*np.pi*(a2**2)*max_a2)

    pa1[a1_sel] = 0
    pa2[a2_sel] = 0

    # redshift
    pz = redshift_pdf(z, z_grid, dVdz_grid, kappa)

    return pm1*pm2*pa1*pa2*pz

def calc_additional_p(declinations: np.ndarray, inclinations: np.ndarray) -> np.ndarray:
    pcost1 = 0.5
    pcost2 = 0.5
    
    # spin azimuths
    pphi1 = 1./(2*np.pi)
    pphi2 = 1./(2*np.pi)
    
    # sky / orientation
    # pcosinc = 0.5
    pinc = 1/2 * np.sin(inclinations)
    pra = 1./(2*np.pi)
    pdec = 0.5*np.cos(declinations)
    ppol = 1./np.pi
    
    p_additional = pcost1*pcost2*pphi1*pphi2*pinc*pra*pdec*ppol
    return p_additional

### Load File

In [4]:
# fname = Path("/hildafs/home/jgassert/hildafs_phy220048p_symlink/share/GWTC-PESamples/posterior_samples/O4/GWTC-4_bbh_posterior_samples_seed1.h5")
fname = Path("/project/ls-gruen/users/julius.gassert/data/pe_samples/IGWN-GWTC2p1-v2-GW150914_095045_PEDataRelease_mixed_nocosmo.h5")

f = h5py.File(fname, "r")
f.keys()
gw_data = pes.io.read(fname)

2026-02-02  15:54:10 PESummary WARNING : Unable to install 'pycbc'. You will not be able to use some of the inbuilt functions.
2026-02-02  15:54:10 PESummary WARNING : Unable to install 'pycbc'. You will not be able to use some of the inbuilt functions.
2026-02-02  15:54:15 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default


In [7]:
samples_dict = gw_data.samples_dict
posterior_samples = samples_dict["C01:Mixed"]
prior_samples = gw_data.priors["samples"]["C01:IMRPhenomXPHM"] # should be the same for Mixed etc according to GWTC info on Zenodo
prior_analytic = gw_data.priors["analytic"]["C01:IMRPhenomXPHM"]

# parameters = posterior_samples.parameters

In [18]:
posterior_samples_Phenom = samples_dict["C01:IMRPhenomXPHM"]
posterior_samples_Phenom["log_likelihood"], posterior_samples_Phenom["log_prior"]

(Array([324.15377634, 322.30427095, 321.9812164 , ...,
        315.18349424, 326.71464775, 322.95371488],
       shape=(199766,)),
 Array([73.61754784, 76.9379596 , 70.77940885, ..., 81.70005815,
        77.18256497, 72.73981805], shape=(199766,)))

In [19]:
posterior_samples_SEOBNR = samples_dict["C01:SEOBNRv4PHM"]
posterior_samples_SEOBNR["log_likelihood"]# , posterior_samples["prior"]

Array([329.04073, 321.63587, 321.49859, ..., 321.76789,
       326.80594, 322.08249], shape=(2239,))

In [27]:
np.sum(np.isin(posterior_samples_SEOBNR["mass_1"], posterior_samples_Phenom["mass_1"]))

np.int64(0)

In [ ]:
prior_model = calc_

In [8]:
logL = posterior_samples["log_likelihood"]         # e.g. array between 310 and 325
logL_shift = logL - np.max(logL)
weights = np.exp(logL_shift)
weights /= np.sum(weights) 

In [9]:
bilby.core.prior.Prior.from_repr?

Object `bilby.core.prior.Prior.from_repr` not found.


In [43]:
import bilby.core.prior as core_prior
from bilby.gw.prior import UniformInComponentsChirpMass, UniformInComponentsMassRatio

core_prior.__dict__["UniformInComponentsChirpMass"] = UniformInComponentsChirpMass
core_prior.__dict__["UniformInComponentsMassRatio"] = UniformInComponentsMassRatio

priors = bilby.core.prior.PriorDict(prior_analytic)

priors_that_matter = []
for prior in priors:
    if not "recalib" in prior:
        print(prior)
        if prior not in posterior_samples.parameters:
            print(f"\n{prior} not in samples.parameters\n")
        else:
            priors_that_matter.append(prior)
priors_that_matter            

a_1
a_2
azimuth

azimuth not in samples.parameters

chirp_mass
geocent_time
luminosity_distance
mass_1
mass_2
mass_ratio
phase
phi_12
phi_jl
psi
theta_jn
tilt_1
tilt_2
time_jitter

time_jitter not in samples.parameters

zenith

zenith not in samples.parameters



['a_1',
 'a_2',
 'chirp_mass',
 'geocent_time',
 'luminosity_distance',
 'mass_1',
 'mass_2',
 'mass_ratio',
 'phase',
 'phi_12',
 'phi_jl',
 'psi',
 'theta_jn',
 'tilt_1',
 'tilt_2']

In [44]:
assert len(priors_that_matter) == 15

In [46]:
total_prior = np.ones_like(posterior_samples["log_likelihood"] )
print(total_prior.shape)

for name in priors_that_matter:
    if "ra" in name or "dec" in name:
        print("Skipping sky location prior multiplication since it is a constant factor (uniform) and we do not care about absolute evidence values.")
        continue
    print(f"{name}: {priors[name]}")
    prior_bilby = priors[name]
    print("Using bilby prior:", prior_bilby)
    print("Using data samples:", posterior_samples[name])
    total_prior *= prior_bilby.prob(posterior_samples[name])

(4478,)
a_1: Uniform(minimum=0, maximum=0.99, name='a_1', latex_label='$a_1$', unit=None, boundary=None)
Using bilby prior: Uniform(minimum=0, maximum=0.99, name='a_1', latex_label='$a_1$', unit=None, boundary=None)
Using data samples: [0.00150978 0.92796454 0.37705213 ... 0.64990346 0.66620969
 0.37417849]
a_2: Uniform(minimum=0, maximum=0.99, name='a_2', latex_label='$a_2$', unit=None, boundary=None)
Using bilby prior: Uniform(minimum=0, maximum=0.99, name='a_2', latex_label='$a_2$', unit=None, boundary=None)
Using data samples: [0.02807403 0.58597864 0.44004783 ... 0.43585748 0.77432163
 0.47331414]
chirp_mass: bilby.gw.prior.UniformInComponentsChirpMass(minimum=21.418182160215295, maximum=41.97447913941358, name='chirp_mass', latex_label='$\\\\mathcal{M}$', unit='$M_{\\\\odot}$', boundary=None)
Using bilby prior: bilby.gw.prior.UniformInComponentsChirpMass(minimum=21.418182160215295, maximum=41.97447913941358, name='chirp_mass', latex_label='$\\\\mathcal{M}$', unit='$M_{\\\\odot}$'

In [47]:
total_prior

Array([3.58550620e-12, 6.51541821e-12, 5.66242208e-12, ...,
       3.32776387e-12, 4.78457278e-13, 4.12443686e-12],
      shape=(4478,))

In [48]:
approximate_posterior = weights * total_prior
approximate_posterior /= np.sum(approximate_posterior)
approximate_posterior

NameError: name 'weights' is not defined

In [49]:
posterior_samples.parameters

['spin_2y',
 'dec',
 'chirp_mass',
 'redshift',
 'theta_jn',
 'ra',
 'a_1',
 'chi_p_2spin',
 'viewing_angle',
 'mass_1_source',
 'cos_tilt_2',
 'spin_2x',
 'geocent_time',
 'mass_2',
 'cos_iota',
 'chi_eff',
 'mass_2_source',
 'psi_J',
 'log_likelihood',
 'symmetric_mass_ratio',
 'phi_12',
 'spin_2z',
 'phase',
 'a_2',
 'beta',
 'spin_1z',
 'chirp_mass_source',
 'mass_ratio',
 'final_mass',
 'comoving_distance',
 'final_mass_source',
 'final_spin',
 'total_mass',
 'mass_1',
 'total_mass_source',
 'chi_p',
 'peak_luminosity',
 'radiated_energy',
 'tilt_1',
 'phi_1',
 'tilt_2',
 'cos_tilt_1',
 'iota',
 'luminosity_distance',
 'phi_2',
 'cos_theta_jn',
 'inverted_mass_ratio',
 'spin_1y',
 'phi_jl',
 'psi',
 'spin_1x',
 'tilt_1_infinity_only_prec_avg',
 'tilt_2_infinity_only_prec_avg',
 'spin_1z_infinity_only_prec_avg',
 'spin_2z_infinity_only_prec_avg',
 'chi_eff_infinity_only_prec_avg',
 'chi_p_infinity_only_prec_avg',
 'cos_tilt_1_infinity_only_prec_avg',
 'cos_tilt_2_infinity_only_prec